# BirdCLEF+ 2026 - V124 Original: Post-Final Rank-Calibrated Tsubasa

Provenance: this notebook continues the license-clean v124/v120-v123 line. It keeps the single-final-layer memory profile, uses the CC0 Tsubasa ConvNeXt SED fold only as a sparse sidecar, and avoids prior output CSVs plus unknown-license SED/cache inputs.

Original experiment: v123's pre-final rank-calibrated sidecar was memory-safe but too muted after the final layer. v124 runs the clean final layer once, then applies a lightweight post-final sidecar correction on the same 10 selected classes when the sidecar rank is ahead of the clean final rank by 0.02.


In [ ]:
!pip install -q --no-deps /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import gc, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import onnxruntime as ort
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
print("ONNX Runtime:", ort.__version__)

In [ ]:
BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
SR = 32000
WINDOW_SAMPLES = SR * 5
N_WINDOWS = 12
CACHE_DIR = None
print("External Perch cache disabled for v124 Snowflake SED EcoProto run")


In [ ]:
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
CLASS_NAME = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA = {"Amphibia", "Insecta"}
print(f"Classes: {N_CLASSES}")

In [ ]:
ONNX_PERCH_PATH = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx")
if not ONNX_PERCH_PATH.exists():
    hits = sorted(Path("/kaggle/input").rglob("perch_v2*.onnx"))
    if not hits:
        raise FileNotFoundError("No Perch ONNX model found")
    ONNX_PERCH_PATH = hits[0]
opts = ort.SessionOptions()
opts.intra_op_num_threads = 4
opts.inter_op_num_threads = 1
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
perch_session = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=opts, providers=["CPUExecutionProvider"])
perch_input_name = perch_session.get_inputs()[0].name
perch_out_map = {o.name: i for i, o in enumerate(perch_session.get_outputs())}
print("Using ONNX Perch:", ONNX_PERCH_PATH)

bc_labels = pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
bc_labels = bc_labels.reset_index().rename(columns={"index": "bc_idx", "inat2024_fsd50k": "sci_name"})
bc_labels["sci_name"] = bc_labels["sci_name"].astype(str)
NO_BC = len(bc_labels)
taxonomy["sci_lookup"] = taxonomy["scientific_name"].astype(str)
merged = taxonomy.merge(bc_labels[["sci_name", "bc_idx"]], left_on="sci_lookup", right_on="sci_name", how="left")
merged["bc_idx"] = merged["bc_idx"].fillna(NO_BC).astype(int)
bc_map = merged.set_index("primary_label")["bc_idx"]
BC_INDICES = np.array([int(bc_map.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED = BC_INDICES != NO_BC
MAPPED_POS = np.where(MAPPED)[0].astype(np.int32)
MAPPED_BC = BC_INDICES[MAPPED].astype(np.int32)
print(f"Mapped: {MAPPED.sum()}/{N_CLASSES}")


In [ ]:
unmapped_amp = merged[(merged["bc_idx"] == NO_BC) & (merged["primary_label"].map(CLASS_NAME) == "Amphibia")]
proxy_map = {}
for _, row in unmapped_amp.iterrows():
    target = row["primary_label"]
    if "son" in str(target): continue
    genus = str(row["scientific_name"]).split()[0]
    hits = bc_labels[bc_labels["sci_name"].str.match(rf"^{re.escape(genus)}\\s", na=False)]
    if len(hits) > 0:
        proxy_map[target] = hits["bc_idx"].astype(int).tolist()
proxy_pos = {label_to_idx[t]: np.array(v, dtype=np.int32) for t, v in proxy_map.items()}
print(f"Frog proxies: {len(proxy_pos)}")

In [ ]:
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\\d+)_(S\\d+)_(\\d{8})_(\\d{6})\\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": None, "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

print("V124: Perch train cache will be rebuilt in-notebook after inference helpers are defined")


In [ ]:
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)

def agg_labels(grp):
    return sorted(set(l.strip() for x in grp for l in str(x).split(";") if l.strip()))

sc = soundscape_labels.groupby(["filename", "start", "end"])["primary_label"].apply(agg_labels).reset_index(name="labels")
sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"] = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)

parsed = sc["filename"].apply(parse_fname).apply(pd.Series)
sc["site"] = parsed["site"]
sc["hour_utc"] = parsed["hour_utc"]

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc["labels"]):
    for l in labels:
        if l in label_to_idx: Y_SC[i, label_to_idx[l]] = 1

win_counts = sc.groupby("filename").size()
full_files = sorted(win_counts[win_counts == N_WINDOWS].index)
sc["full"] = sc["filename"].isin(full_files)
full_rows = sc[sc["full"]].sort_values(["filename", "end_sec"]).reset_index(drop=False)
Y_FULL = Y_SC[full_rows["index"].to_numpy()]
print(f"Full files: {len(full_files)}, Windows: {len(full_rows)}, Active: {(Y_FULL.sum(0) > 0).sum()}")

In [ ]:
# V124: audited CC0 Tsubasa Snowflake SED ensemble.
SNOWFLAKE_ROOT = Path("/kaggle/input/datasets/tsubasatech/birdclef-2026-snowflake-sed")
SNOWFLAKE_MODEL_NAMES = [
    "sed_convnext-tiny_fold0.onnx",
]
CLEAN_SED_PATHS = []
for model_name in SNOWFLAKE_MODEL_NAMES:
    cand = SNOWFLAKE_ROOT / model_name
    if not cand.exists():
        hits = sorted(Path("/kaggle/input").rglob(model_name))
        if not hits:
            raise FileNotFoundError(f"No Snowflake SED ONNX model found: {model_name}")
        cand = hits[0]
    CLEAN_SED_PATHS.append(cand)

sed_opts = ort.SessionOptions()
sed_opts.intra_op_num_threads = 4
sed_opts.inter_op_num_threads = 1
sed_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
clean_sed_sessions = []
for sed_path in CLEAN_SED_PATHS:
    sess = ort.InferenceSession(str(sed_path), sess_options=sed_opts, providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    out_map = {o.name: i for i, o in enumerate(sess.get_outputs())}
    clean_sed_sessions.append((sess, input_name, out_map, sed_path))
    print("Using V124 Snowflake SED:", sed_path)
    print("Snowflake SED inputs:", [(x.name, x.shape, x.type) for x in sess.get_inputs()])
    print("Snowflake SED outputs:", [(x.name, x.shape, x.type) for x in sess.get_outputs()])
print(f"V124 Tsubasa ConvNeXt SED ensemble size: {len(clean_sed_sessions)}")


In [ ]:
import concurrent.futures

def read_audio(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(1)
    if sr != SR: raise ValueError(f"Bad SR {sr}")
    target = SR * 60
    if len(y) < target: y = np.pad(y, (0, target - len(y)))
    return y[:target]

def _find_embedding_output(outs):
    target_dim = 1536
    for idx, arr in enumerate(outs):
        if getattr(arr, "ndim", 0) == 2 and arr.shape[1] == target_dim:
            return idx
    return None

def _features2d(arr, max_flat=4096):
    """Convert an ONNX feature output to a compact 2-D matrix for ridge probes."""
    a = np.asarray(arr, dtype=np.float32)
    if a.ndim == 2:
        return a
    if a.ndim >= 3 and a.shape[-1] <= max_flat:
        z = a.reshape(a.shape[0], -1, a.shape[-1])
        return np.concatenate([z.mean(axis=1), z.std(axis=1), z.max(axis=1)], axis=1).astype(np.float32)
    flat = a.reshape(a.shape[0], -1)
    if flat.shape[1] > max_flat:
        stride = int(np.ceil(flat.shape[1] / max_flat))
        flat = flat[:, ::stride]
    return flat.astype(np.float32)

def run_perch_fast(paths, batch_files=16, capture_spatial=True, desc="ONNX Perch inference"):
    """Batched ONNX Perch inference over 12 five-second windows per file.
    V73 captures both embedding and compact spatial_embedding features when exposed.
    """
    paths = [Path(p) for p in paths]
    rows = len(paths) * N_WINDOWS
    row_ids = np.empty(rows, object); fnames = np.empty(rows, object)
    scores = np.zeros((rows, N_CLASSES), np.float32)
    test_emb = None
    spatial_feat = None
    emb_output_idx = None
    spatial_output_idx = None
    w = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        for start in tqdm(range(0, len(paths), batch_files), desc=desc):
            batch_paths = paths[start:start + batch_files]
            audio_batch = list(io_executor.map(read_audio, batch_paths))
            x = np.empty((len(batch_paths) * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = w
            for bi, path in enumerate(batch_paths):
                audio = audio_batch[bi]
                x[bi*N_WINDOWS:(bi+1)*N_WINDOWS] = audio.reshape(N_WINDOWS, WINDOW_SAMPLES)
                stem = path.stem
                row_ids[w:w+N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                fnames[w:w+N_WINDOWS] = path.name
                w += N_WINDOWS
            outs = perch_session.run(None, {perch_input_name: x})
            if emb_output_idx is None:
                emb_output_idx = _find_embedding_output(outs)
                spatial_output_idx = perch_out_map.get("spatial_embedding") if capture_spatial else None
                print("Perch output names:", list(perch_out_map.keys()))
                print("Detected embedding output index:", emb_output_idx)
                print("Detected spatial_embedding output index:", spatial_output_idx)
                if emb_output_idx is not None:
                    test_emb = np.zeros((rows, 1536), np.float32)
                if spatial_output_idx is not None:
                    first_spatial = _features2d(outs[spatial_output_idx])
                    spatial_feat = np.zeros((rows, first_spatial.shape[1]), np.float32)
                    spatial_feat[br:w] = first_spatial
                    print("Spatial feature dim:", first_spatial.shape[1])
            logits = outs[perch_out_map["label"]].astype(np.float32)
            scores[br:w, MAPPED_POS] = logits[:, MAPPED_BC]
            for pos, bc_arr in proxy_pos.items():
                scores[br:w, pos] = logits[:, bc_arr].max(1).astype(np.float32)
            if emb_output_idx is not None:
                test_emb[br:w] = outs[emb_output_idx].astype(np.float32)
            if spatial_output_idx is not None and spatial_feat is not None and spatial_feat[br:w].sum() == 0:
                spatial_feat[br:w] = _features2d(outs[spatial_output_idx])
            del x, logits, outs, audio_batch
            gc.collect()
    return pd.DataFrame({"row_id": row_ids, "filename": fnames}), scores, test_emb, spatial_feat



def run_clean_sed_fast(paths, batch_files=16, desc="Snowflake SED inference"):
    """Run the audited CC0 raw-audio SED ONNX model over 12 five-second windows."""
    paths = [Path(p) for p in paths]
    rows = len(paths) * N_WINDOWS
    row_ids = np.empty(rows, object)
    fnames = np.empty(rows, object)
    logits_out = np.zeros((rows, N_CLASSES), np.float32)
    w = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        for start in tqdm(range(0, len(paths), batch_files), desc=desc):
            batch_paths = paths[start:start + batch_files]
            audio_batch = list(io_executor.map(read_audio, batch_paths))
            x = np.empty((len(batch_paths) * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = w
            for bi, path in enumerate(batch_paths):
                audio = audio_batch[bi]
                x[bi*N_WINDOWS:(bi+1)*N_WINDOWS] = audio.reshape(N_WINDOWS, WINDOW_SAMPLES)
                stem = path.stem
                row_ids[w:w+N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                fnames[w:w+N_WINDOWS] = path.name
                w += N_WINDOWS
            logits_acc = None
            for sess, input_name, out_map, sed_path in clean_sed_sessions:
                outs = sess.run(None, {input_name: x})
                out_idx = out_map.get("clip_logits", out_map.get("logits", 0))
                logits = outs[out_idx].astype(np.float32)
                if logits.shape[1] != N_CLASSES:
                    raise ValueError(f"Snowflake SED class mismatch for {sed_path}: {logits.shape[1]} vs {N_CLASSES}")
                logits_acc = logits if logits_acc is None else logits_acc + logits
            logits = (logits_acc / max(1, len(clean_sed_sessions))).astype(np.float32)
            logits_out[br:w] = logits
            del x, logits, logits_acc, outs, audio_batch
            gc.collect()
    return pd.DataFrame({"row_id": row_ids, "filename": fnames}), logits_out


In [ ]:
# V124: rebuild train Perch and Snowflake SED features in-notebook.
train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
train_paths = [p for p in train_paths if p.exists()]
print(f"V124 rebuilding Perch train features for {len(train_paths)} files")
meta_full, scores_full_raw, emb_full, _spatial_train_unused = run_perch_fast(
    train_paths, capture_spatial=False, desc="V124 Perch train feature rebuild"
)
meta_full = meta_full.set_index("row_id").loc[full_rows["row_id"].astype(str)].reset_index()
scores_full_raw = scores_full_raw[: len(meta_full)]
emb_full = emb_full[: len(meta_full)]
assert np.all(meta_full["filename"].values == full_rows["filename"].values)
print(f"V124 rebuilt train features: meta={meta_full.shape}, scores={scores_full_raw.shape}, emb={emb_full.shape}")

clean_sed_meta_full, clean_sed_train_logits = run_clean_sed_fast(
    train_paths, desc="V124 Snowflake SED train feature rebuild"
)
clean_sed_meta_full = clean_sed_meta_full.set_index("row_id").loc[full_rows["row_id"].astype(str)].reset_index()
clean_sed_train_logits = clean_sed_train_logits[: len(clean_sed_meta_full)]
assert np.all(clean_sed_meta_full["filename"].values == full_rows["filename"].values)
print(f"V124 rebuilt Snowflake SED train logits: meta={clean_sed_meta_full.shape}, logits={clean_sed_train_logits.shape}")


In [ ]:
# Build OOF predictions using GroupKFold
# For each fold: train on other files, predict on held-out files
# Use raw Perch scores + simple priors for unmapped classes

meta_aligned = meta_full.set_index("row_id").loc[full_rows["row_id"]].reset_index()
assert np.all(meta_aligned["filename"].values == full_rows["filename"].values)

n_splits = 5
gkf = GroupKFold(n_splits)
groups = meta_aligned["filename"].to_numpy()

oof_logits = np.zeros_like(scores_full_raw)  # Will store OOF logits

for fold, (tr, va) in enumerate(gkf.split(scores_full_raw, groups=groups)):
    # For mapped classes: use raw Perch logits directly as OOF
    oof_logits[np.ix_(va, MAPPED_POS)] = scores_full_raw[np.ix_(va, MAPPED_POS)]
    # For unmapped non-proxy classes: use global prior as fallback
    global_prior = Y_SC[tr].mean(0).astype(np.float32)
    for i in np.where(~MAPPED)[0]:
        if i not in proxy_pos:
            # Convert prior probability to logit
            p = np.clip(global_prior[i], 1e-4, 1 - 1e-4)
            oof_logits[va, i] = np.log(p / (1 - p))
    # For proxy classes: use max of proxy Perch scores
    for pos, bc_arr in proxy_pos.items():
        oof_logits[va, pos] = scores_full_raw[np.ix_(va, bc_arr)].max(1)

print(f"OOF logits shape: {oof_logits.shape}")

In [ ]:
# Evaluate raw Perch OOF performance
oof_probs_raw = 1.0 / (1.0 + np.exp(-oof_logits))
keep = Y_FULL.sum(0) > 0
auc_raw = roc_auc_score(Y_FULL[:, keep], oof_probs_raw[:, keep], average='macro')
print(f"Raw Perch OOF AUC: {auc_raw:.4f}")

# Per-class AUC to identify weak classes
per_class_auc = []
for ci in range(N_CLASSES):
    if Y_FULL[:, ci].sum() == 0 or Y_FULL[:, ci].sum() == len(Y_FULL):
        per_class_auc.append(np.nan)
    else:
        per_class_auc.append(roc_auc_score(Y_FULL[:, ci], oof_probs_raw[:, ci]))
per_class_auc = np.array(per_class_auc)
print(f"Per-class AUC: mean={np.nanmean(per_class_auc):.4f}, min={np.nanmin(per_class_auc):.4f}, max={np.nanmax(per_class_auc):.4f}")
print(f"Classes with AUC < 0.5: {(per_class_auc < 0.5).sum()}")

In [ ]:
# OOF-based Platt scaling per class
# Fit logistic regression on OOF logits to calibrate probabilities

calibrators = {}
for ci in tqdm(range(N_CLASSES), desc="Fitting calibrators"):
    y = Y_FULL[:, ci]
    if y.sum() == 0 or y.sum() == len(y): continue
    # Use OOF logits as feature, fit Platt scaling
    X = oof_logits[:, ci].reshape(-1, 1)
    clf = LogisticRegression(C=1.0, max_iter=100, solver='lbfgs')
    clf.fit(X, y)
    calibrators[ci] = clf

print(f"Fitted {len(calibrators)} calibrators")

# Apply calibration to OOF and evaluate
oof_probs_cal = oof_probs_raw.copy()
for ci, clf in calibrators.items():
    oof_probs_cal[:, ci] = clf.predict_proba(oof_logits[:, ci].reshape(-1, 1))[:, 1]

auc_cal = roc_auc_score(Y_FULL[:, keep], oof_probs_cal[:, keep], average='macro')
print(f"Calibrated OOF AUC: {auc_cal:.4f} (delta: {auc_cal - auc_raw:+.4f})")

per_class_auc_cal = []
for ci in range(N_CLASSES):
    y = Y_FULL[:, ci]
    if y.sum() == 0 or y.sum() == len(y):
        per_class_auc_cal.append(np.nan)
    else:
        per_class_auc_cal.append(roc_auc_score(y, oof_probs_cal[:, ci]))
per_class_auc_cal = np.array(per_class_auc_cal, dtype=np.float32)
print(f"Calibrated per-class AUC: mean={np.nanmean(per_class_auc_cal):.4f}, min={np.nanmin(per_class_auc_cal):.4f}, max={np.nanmax(per_class_auc_cal):.4f}")

In [ ]:
# Per-class threshold optimization on OOF
# For each class, find threshold that maximizes F1

def f1_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tp = (y_pred * y_true).sum()
    fp = (y_pred * (1 - y_true)).sum()
    fn = ((1 - y_pred) * y_true).sum()
    if tp == 0: return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)

thresholds = np.full(N_CLASSES, 0.5, dtype=np.float32)
for ci in range(N_CLASSES):
    y = Y_FULL[:, ci]
    if y.sum() == 0 or y.sum() == len(y): continue
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.05):
        f1 = f1_at_threshold(y, oof_probs_cal[:, ci], t)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    thresholds[ci] = best_t

print(f"Thresholds: mean={thresholds.mean():.3f}, min={thresholds.min():.3f}, max={thresholds.max():.3f}")

In [ ]:
test_dir = BASE / "test_soundscapes"
test_paths = sorted(test_dir.glob("*.ogg"))
if len(test_paths) == 0:
    print("No test files - using train soundscapes for dry run")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:10]
print(f"Processing {len(test_paths)} files")
meta_test, scores_test, emb_test, spatial_test = run_perch_fast(test_paths)
print("Test embeddings:", None if emb_test is None else emb_test.shape)
print("Test spatial features:", None if spatial_test is None else spatial_test.shape)
clean_sed_meta_test, clean_sed_test_logits = run_clean_sed_fast(test_paths, desc="V124 Snowflake SED test inference")
assert clean_sed_meta_test["row_id"].astype(str).tolist() == meta_test["row_id"].astype(str).tolist(), "Snowflake SED row order mismatch"
print("Snowflake SED test logits:", clean_sed_test_logits.shape)



In [ ]:
# V124 original branch: audited CC0 raw-audio SED with train-window column remap.
raw_clean_sed_train_probs = (1.0 / (1.0 + np.exp(-np.clip(clean_sed_train_logits, -50, 50)))).astype(np.float32)
raw_clean_sed_test_probs = (1.0 / (1.0 + np.exp(-np.clip(clean_sed_test_logits, -50, 50)))).astype(np.float32)

def _rank_rows_local(values, power=1.0):
    v = np.asarray(values, dtype=np.float32)
    order = np.argsort(v, axis=1)
    ranks = np.empty_like(v, dtype=np.float32)
    grid = np.linspace(0.0, 1.0, v.shape[1], dtype=np.float32)
    ranks[np.arange(v.shape[0])[:, None], order] = grid
    if power != 1.0:
        ranks = np.power(np.clip(ranks, 0.0, 1.0), power).astype(np.float32)
    return ranks

def _column_corr_matrix(x, y):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    xc = x - x.mean(axis=0, keepdims=True)
    yc = y - y.mean(axis=0, keepdims=True)
    denom = (np.sqrt((xc * xc).sum(axis=0))[:, None] * np.sqrt((yc * yc).sum(axis=0))[None, :]) + 1e-6
    return (xc.T @ yc / denom).astype(np.float32)

def _safe_auc(y, score):
    if y.sum() == 0 or y.sum() == len(y):
        return np.nan
    return float(roc_auc_score(y, score))

col_corr = _column_corr_matrix(raw_clean_sed_train_probs, Y_FULL)
identity_cols = np.arange(N_CLASSES, dtype=np.int32)
remap_cols = identity_cols.copy()
remap_strength = np.zeros(N_CLASSES, dtype=np.float32)
sed_same_auc = np.full(N_CLASSES, np.nan, dtype=np.float32)
sed_best_auc = np.full(N_CLASSES, np.nan, dtype=np.float32)
sed_same_corr = np.full(N_CLASSES, np.nan, dtype=np.float32)
sed_best_corr = np.full(N_CLASSES, np.nan, dtype=np.float32)

for ci in range(N_CLASSES):
    y = Y_FULL[:, ci]
    if y.sum() < 2 or (len(y) - y.sum()) < 2:
        continue
    same_j = ci
    same_auc = _safe_auc(y, raw_clean_sed_train_probs[:, same_j])
    same_corr = float(col_corr[same_j, ci])
    cand = np.argsort(col_corr[:, ci])[-8:][::-1]
    best_j = same_j
    best_auc = same_auc
    best_corr = same_corr
    for jj in cand:
        auc_j = _safe_auc(y, raw_clean_sed_train_probs[:, jj])
        corr_j = float(col_corr[jj, ci])
        if np.isfinite(auc_j) and (not np.isfinite(best_auc) or auc_j > best_auc + 1e-6):
            best_j = int(jj)
            best_auc = float(auc_j)
            best_corr = corr_j
    sed_same_auc[ci] = same_auc
    sed_best_auc[ci] = best_auc
    sed_same_corr[ci] = same_corr
    sed_best_corr[ci] = best_corr
    auc_gain = (best_auc - same_auc) if np.isfinite(best_auc) and np.isfinite(same_auc) else 0.0
    corr_gain = best_corr - same_corr
    if (
        best_j != same_j
        and best_auc >= max(0.58, same_auc + 0.055)
        and best_corr >= max(0.035, same_corr + 0.012)
        and y.sum() >= 2
    ):
        remap_cols[ci] = best_j
        remap_strength[ci] = np.clip(0.30 + 2.60 * max(0.0, auc_gain) + 1.20 * max(0.0, corr_gain), 0.30, 0.88)

same_view_train = raw_clean_sed_train_probs
remap_view_train = raw_clean_sed_train_probs[:, remap_cols]
same_view_test = raw_clean_sed_test_probs
remap_view_test = raw_clean_sed_test_probs[:, remap_cols]
strength_2d = remap_strength.reshape(1, -1).astype(np.float32)
sed_train_view = ((1.0 - strength_2d) * same_view_train + strength_2d * remap_view_train).astype(np.float32)
sed_test_view = ((1.0 - strength_2d) * same_view_test + strength_2d * remap_view_test).astype(np.float32)

# Stabilize the Snowflake SED branch for AUC-style use: keep calibrated probabilities
# but inject row-wise rank information only where the remap earned trust.
rank_view = _rank_rows_local(sed_test_view, power=0.74)
rank_mix = (0.18 + 0.42 * remap_strength).reshape(1, -1).astype(np.float32)
sed_probs = ((1.0 - rank_mix) * sed_test_view + rank_mix * rank_view).astype(np.float32)
sed_rows = meta_test["row_id"].astype(str).tolist()
assert sed_rows == meta_test["row_id"].astype(str).tolist(), "Snowflake SED row_id order mismatch"

sed_per_class_auc = []
for ci in range(N_CLASSES):
    sed_per_class_auc.append(_safe_auc(Y_FULL[:, ci], sed_train_view[:, ci]))
sed_per_class_auc = np.array(sed_per_class_auc, dtype=np.float32)

remap_diag = pd.DataFrame({
    "primary_label": PRIMARY_LABELS,
    "support": Y_FULL.sum(axis=0).astype(int),
    "same_col": identity_cols,
    "best_col": remap_cols,
    "remapped": remap_cols != identity_cols,
    "strength": remap_strength,
    "same_auc": sed_same_auc,
    "best_auc": sed_best_auc,
    "same_corr": sed_same_corr,
    "best_corr": sed_best_corr,
})
remap_diag.to_csv("v124_postfinal_tsubasa_remap_diagnostics.csv", index=False)
print(f"V124 Snowflake SED remap view: shape={sed_probs.shape}, range=[{sed_probs.min():.6f}, {sed_probs.max():.6f}]")
print(f"V124 Snowflake SED remaps accepted: {int((remap_cols != identity_cols).sum())}/{N_CLASSES}; strength mean/max={remap_strength.mean():.4f}/{remap_strength.max():.4f}")
print(f"V124 Snowflake SED train AUC ref: mean={np.nanmean(sed_per_class_auc):.4f}, min={np.nanmin(sed_per_class_auc):.4f}, active={(np.isfinite(sed_per_class_auc)).sum()}")
print("V124 remap diagnostics saved: v124_postfinal_tsubasa_remap_diagnostics.csv")


# V124 keeps a lightweight rank-calibrated Tsubasa sidecar for post-final use,
# but feeds the clean Perch surrogate into the expensive final layer so that
# the final layer runs exactly once.
v124_tsubasa_sed_probs = sed_probs.copy()
v124_clean_sed_probs = (1.0 / (1.0 + np.exp(-np.clip(scores_test, -50, 50)))).astype(np.float32)
v124_clean_sed_auc = np.nan_to_num(per_class_auc_cal, nan=0.50).astype(np.float32)

def _v124_rank_cols(values):
    return pd.DataFrame(np.asarray(values, dtype=np.float32)).rank(axis=0, pct=True).to_numpy(np.float32)

v124_tsubasa_rank = _v124_rank_cols(v124_tsubasa_sed_probs)
v124_tsubasa_rankcal = np.empty_like(v124_clean_sed_probs, dtype=np.float32)
for _ci in range(N_CLASSES):
    _side_order = np.argsort(v124_tsubasa_rank[:, _ci])
    _clean_sorted = np.sort(v124_clean_sed_probs[:, _ci]).astype(np.float32)
    v124_tsubasa_rankcal[_side_order, _ci] = _clean_sorted

v124_selected_classes = ["47158son13", "47158son15", "47158son16", "47158son21", "47158son22", "47158son23", "516975", "chacha1", "grekis", "plcjay1"]
v124_selected_set = set(v124_selected_classes)
v124_selected_mask = np.array([label in v124_selected_set for label in PRIMARY_LABELS], dtype=bool)

sed_probs = v124_clean_sed_probs.astype(np.float32)
sed_per_class_auc = v124_clean_sed_auc.astype(np.float32)
print(f"V124 stored rank-calibrated sidecar for post-final use: range=[{v124_tsubasa_rankcal.min():.6f}, {v124_tsubasa_rankcal.max():.6f}]")
print("V124 final layer input: clean Perch surrogate; final layer will run once")


In [ ]:
# V68 original final layer: balanced rank consensus with Perch safety and softened spike preservation.
# This replaces the v20 final postprocess only; upstream Perch/SED inference remains the private original fast-ONNX path.
perch_probs = scores_test.copy()

for ci, clf in calibrators.items():
    perch_probs[:, ci] = clf.predict_proba(scores_test[:, ci].reshape(-1, 1))[:, 1]
for ci in range(N_CLASSES):
    if ci not in calibrators:
        perch_probs[:, ci] = 1.0 / (1.0 + np.exp(-scores_test[:, ci]))

thr = thresholds.reshape(1, -1).astype(np.float32)
soft_factor = np.clip(0.82 - 0.82 * thresholds, 0.18, 0.72).reshape(1, -1).astype(np.float32)
below_thr = perch_probs < thr
perch_probs = np.where(below_thr, perch_probs * soft_factor, perch_probs).astype(np.float32)
print(f"Perch classwise soft suppression: {below_thr.mean():.3f}, factor=[{soft_factor.min():.3f},{soft_factor.max():.3f}]")

def _rank01(arr):
    return pd.DataFrame(np.clip(arr, 1e-7, 1 - 1e-7)).rank(axis=0, pct=True).to_numpy(np.float32)

def _rank_power(arr, gamma):
    r = _rank01(arr)
    return np.power(np.clip(r, 1e-6, 1.0), gamma).astype(np.float32)

def _safe_auc(arr, fallback):
    out = arr.astype(np.float32).copy()
    out[~np.isfinite(out)] = fallback
    return np.clip(out, 0.01, 0.99)

def _blend_rank(sed_weight):
    sed_weight = sed_weight.astype(np.float32)
    return ((1.0 - sed_weight).reshape(1, -1) * perch_rank + sed_weight.reshape(1, -1) * sed_rank).astype(np.float32)

def _normalize_prior_key(key):
    if not isinstance(key, tuple):
        key = (key,)
    return tuple("__missing__" if pd.isna(v) else v for v in key)

def _build_prior_lookup(df, y_mat, key_cols, alpha=6.0):
    global_prior = np.clip(y_mat.mean(axis=0).astype(np.float32), 1e-4, 1 - 1e-4)
    clean = df[key_cols].copy()
    for col in key_cols:
        clean[col] = clean[col].where(clean[col].notna(), "__missing__")
    lookup, counts = {}, {}
    for key, idx in clean.groupby(key_cols, sort=False).indices.items():
        key = _normalize_prior_key(key)
        idx = np.asarray(idx, dtype=np.int32)
        counts[key] = int(len(idx))
        lookup[key] = ((y_mat[idx].sum(axis=0) + alpha * global_prior) / (len(idx) + alpha)).astype(np.float32)
    return lookup, counts, global_prior

def _site_hour_prior_for_test(meta_df):
    train_meta = full_rows[["site", "hour_utc"]].copy().reset_index(drop=True)
    sh_lookup, sh_counts, global_prior = _build_prior_lookup(train_meta, Y_FULL, ["site", "hour_utc"], alpha=10.0)
    s_lookup, s_counts, _ = _build_prior_lookup(train_meta, Y_FULL, ["site"], alpha=12.0)
    h_lookup, h_counts, _ = _build_prior_lookup(train_meta, Y_FULL, ["hour_utc"], alpha=16.0)
    parsed = meta_df["filename"].apply(parse_fname).apply(pd.Series)
    priors = np.empty((len(meta_df), N_CLASSES), dtype=np.float32)
    strength = np.empty(len(meta_df), dtype=np.float32)
    for i, row in parsed.iterrows():
        site_raw = row.get("site", None)
        site = "__missing__" if pd.isna(site_raw) else site_raw
        hour_raw = row.get("hour_utc", -1)
        hour = int(hour_raw) if pd.notna(hour_raw) else -1
        sh_key, s_key, h_key = _normalize_prior_key((site, hour)), _normalize_prior_key((site,)), _normalize_prior_key((hour,))
        if sh_key in sh_lookup and sh_counts.get(sh_key, 0) >= 2:
            priors[i] = sh_lookup[sh_key]; strength[i] = min(1.0, sh_counts[sh_key] / 12.0)
        elif s_key in s_lookup:
            priors[i] = s_lookup[s_key]; strength[i] = min(0.70, s_counts.get(s_key, 0) / 20.0)
        elif h_key in h_lookup:
            priors[i] = h_lookup[h_key]; strength[i] = min(0.48, h_counts.get(h_key, 0) / 28.0)
        else:
            priors[i] = global_prior; strength[i] = 0.20
    return np.clip(priors, 1e-4, 1 - 1e-4), strength

perch_rank = _rank01(perch_probs)
sed_rank = _rank01(sed_probs)
perch_auc_raw_ref = _safe_auc(per_class_auc.astype(np.float32), float(np.nanmean(per_class_auc)))
perch_auc_cal_ref = _safe_auc(per_class_auc_cal, float(np.nanmean(per_class_auc_cal)))
sed_auc_ref = _safe_auc(sed_per_class_auc, float(np.nanmean(sed_per_class_auc)))
support = Y_FULL.sum(axis=0).astype(np.float32)
low_support = support < 3

# Three original views. Public leaderboard patterns only motivate the search direction; the formulas are written here.
auc_delta = np.clip(sed_auc_ref - perch_auc_cal_ref, -0.30, 0.30)
support_shrink = np.clip((support - 1.0) / 10.0, 0.15, 1.0).astype(np.float32)

sed_w_sedfirst = np.where(MAPPED, 0.62, 0.74).astype(np.float32)
sed_w_sedfirst[low_support & MAPPED] = 0.54
sed_w_sedfirst[low_support & (~MAPPED)] = 0.68

sed_w_adaptive = np.where(MAPPED, 0.55, 0.68).astype(np.float32) + 0.22 * auc_delta.astype(np.float32) * support_shrink
sed_w_adaptive[sed_auc_ref < 0.55] -= 0.10
sed_w_adaptive[perch_auc_cal_ref < 0.55] += 0.06
sed_w_adaptive = np.clip(sed_w_adaptive, 0.38, 0.80).astype(np.float32)

sed_w_safety = np.where(MAPPED, 0.47, 0.60).astype(np.float32)
sed_w_safety[low_support & MAPPED] = 0.46
sed_w_safety[low_support & (~MAPPED)] = 0.62

view_sedfirst = _rank_power(_blend_rank(sed_w_sedfirst), 0.92)
view_adaptive = _rank_power(_blend_rank(sed_w_adaptive), 0.98)
view_safety = _rank_power(_blend_rank(sed_w_safety), 1.04)
view_perch = _rank_power(0.70 * perch_probs + 0.30 * perch_rank, 1.02)

# V73: original dual ridge probes. Embedding uses cached train features; spatial features are recomputed in-kernel.
def _cache_positions_for_full_rows():
    meta_order = meta_full.reset_index(drop=True).reset_index().rename(columns={"index": "cache_pos"})
    if "row_id" not in meta_order.columns:
        raise ValueError("perch-meta cache does not expose row_id for embedding alignment")
    return meta_order.set_index("row_id").loc[full_rows["row_id"].astype(str), "cache_pos"].to_numpy()

def _ridge_probe_rank(x_train_raw, x_test_raw, label, lam=18.0, temp=0.85, gamma=1.0):
    x_train = x_train_raw.astype(np.float32)
    x_test = x_test_raw.astype(np.float32)
    mu = x_train.mean(axis=0, keepdims=True)
    sd = x_train.std(axis=0, keepdims=True) + 1e-5
    x_train = np.clip((x_train - mu) / sd, -5.0, 5.0).astype(np.float32)
    x_test = np.clip((x_test - mu) / sd, -5.0, 5.0).astype(np.float32)
    y = Y_FULL.astype(np.float32)
    y_center = y - y.mean(axis=0, keepdims=True)
    k = x_train @ x_train.T
    k.flat[::k.shape[0]+1] += lam
    alpha = np.linalg.solve(k.astype(np.float32), y_center.astype(np.float32))
    w = x_train.T @ alpha
    logits_probe = x_test @ w + y.mean(axis=0, keepdims=True)
    logits_probe = (logits_probe - logits_probe.mean(axis=0, keepdims=True)) / (logits_probe.std(axis=0, keepdims=True) + 1e-5)
    probe_prob = 1.0 / (1.0 + np.exp(-temp * logits_probe))
    probe_rank = _rank_power(probe_prob, gamma)
    spread = float(np.nanstd(probe_rank))
    if not np.isfinite(spread) or spread < 0.05:
        print(f"{label} probe rejected, spread={spread}")
        return view_perch, 0.0
    print(f"{label} probe active: train={x_train.shape}, test={x_test.shape}, rank std={spread}")
    return probe_rank.astype(np.float32), spread

def _embedding_probe_rank():
    if emb_test is None:
        print("Embedding probe unavailable: no test embedding output")
        return view_perch, 0.0
    cache_pos = _cache_positions_for_full_rows()
    x_train = emb_full.astype(np.float32)[cache_pos]
    print("Embedding probe aligned train rows:", x_train.shape, "unique cache rows", len(np.unique(cache_pos)))
    return _ridge_probe_rank(x_train, emb_test, "Embedding", lam=18.0, temp=0.85, gamma=1.00)

def _mlp_embedding_probe_rank():
    if emb_test is None:
        print("MLP embedding probe unavailable: no test embedding output")
        return view_perch, 0.0
    cache_pos = _cache_positions_for_full_rows()
    x_train_raw = emb_full.astype(np.float32)[cache_pos]
    x_test_raw = emb_test.astype(np.float32)
    try:
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train_raw).astype(np.float32)
        x_test = scaler.transform(x_test_raw).astype(np.float32)
        n_comp = int(min(96, x_train.shape[0] - 1, x_train.shape[1]))
        pca = PCA(n_components=n_comp, random_state=20260518, svd_solver="randomized")
        x_train = pca.fit_transform(x_train).astype(np.float32)
        x_test = pca.transform(x_test).astype(np.float32)
        clf = MLPClassifier(hidden_layer_sizes=(96,), activation="relu", solver="adam", alpha=0.012, batch_size=128, learning_rate_init=0.002, max_iter=70, early_stopping=True, validation_fraction=0.12, n_iter_no_change=8, random_state=20260518, verbose=False)
        clf.fit(x_train, Y_FULL.astype(np.int8))
        prob = clf.predict_proba(x_test)
        if isinstance(prob, list):
            prob = np.stack([p[:, -1] if p.ndim == 2 else p for p in prob], axis=1)
        prob = np.asarray(prob, dtype=np.float32)
        if prob.shape != (x_test.shape[0], N_CLASSES):
            print("MLP embedding probe rejected: shape", prob.shape)
            return view_perch, 0.0
        rank = _rank_power(np.clip(prob, 1e-6, 1.0 - 1e-6), 1.0)
        spread = float(np.nanstd(rank))
        if not np.isfinite(spread) or spread < 0.05:
            print(f"MLP embedding probe rejected, spread={spread}")
            return view_perch, 0.0
        print(f"MLP embedding probe active: train={x_train.shape}, test={x_test.shape}, n_iter={getattr(clf, 'n_iter_', None)}, rank std={spread}")
        return rank.astype(np.float32), spread
    except Exception as exc:
        print("MLP embedding probe failed:", repr(exc))
        return view_perch, 0.0

def _train_spatial_features():
    if spatial_test is None:
        return None
    train_names = list(dict.fromkeys(full_rows["filename"].astype(str).tolist()))
    train_paths = [BASE / "train_soundscapes" / name for name in train_names]
    train_paths = [p for p in train_paths if p.exists()]
    if len(train_paths) == 0:
        print("Spatial probe unavailable: no train soundscape files")
        return None
    meta_sp, _, _, spatial_full_raw = run_perch_fast(train_paths, batch_files=16, capture_spatial=True, desc="ONNX Perch train spatial")
    if spatial_full_raw is None:
        print("Spatial probe unavailable: no spatial train features")
        return None
    sp_order = meta_sp.reset_index(drop=True).reset_index().rename(columns={"index": "cache_pos"})
    pos = sp_order.set_index("row_id").loc[full_rows["row_id"].astype(str), "cache_pos"].to_numpy()
    x_train = spatial_full_raw[pos]
    print("Spatial probe aligned train rows:", x_train.shape, "unique spatial rows", len(np.unique(pos)))
    return x_train.astype(np.float32)

def _dual_probe_rank():
    emb_rank, emb_spread = _embedding_probe_rank()
    mlp_rank, mlp_spread = _mlp_embedding_probe_rank()
    spatial_rank, spatial_spread = view_perch, 0.0
    x_spatial_train = _train_spatial_features()
    if x_spatial_train is not None and spatial_test is not None:
        spatial_rank, spatial_spread = _ridge_probe_rank(x_spatial_train, spatial_test, "Spatial", lam=26.0, temp=0.70, gamma=1.02)
    if emb_spread > 0 and spatial_spread > 0 and mlp_spread > 0:
        combined = (0.64 * emb_rank + 0.26 * spatial_rank + 0.10 * mlp_rank).astype(np.float32)
        print(f"Dual probe active: embedding=0.64 spatial=0.26 mlp=0.10 combined std={float(np.nanstd(combined))}")
        return combined, float(np.nanstd(combined))
    if emb_spread > 0 and spatial_spread > 0:
        combined = (0.72 * emb_rank + 0.28 * spatial_rank).astype(np.float32)
        print(f"Dual probe active: embedding=0.72 spatial=0.28 combined std={float(np.nanstd(combined))}")
        return combined, float(np.nanstd(combined))
    if emb_spread > 0 and mlp_spread > 0:
        combined = (0.88 * emb_rank + 0.12 * mlp_rank).astype(np.float32)
        print(f"Dual probe fallback: embedding=0.88 mlp=0.12 combined std={float(np.nanstd(combined))}")
        return combined, float(np.nanstd(combined))
    if emb_spread > 0:
        print("Dual probe fallback: embedding only")
        return emb_rank, emb_spread
    if spatial_spread > 0:
        print("Dual probe fallback: spatial only")
        return spatial_rank, spatial_spread
    print("Dual probe unavailable: fallback to Perch view")
    return view_perch, 0.0

view_probe, probe_spread = _dual_probe_rank()

if probe_spread > 0:
    final_probs = (
        0.30 * view_sedfirst +
        0.25 * view_adaptive +
        0.19 * view_safety +
        0.11 * view_perch +
        0.15 * view_probe
    ).astype(np.float32)
else:
    final_probs = (
        0.34 * view_sedfirst +
        0.28 * view_adaptive +
        0.22 * view_safety +
        0.16 * view_perch
    ).astype(np.float32)

# Small support-aware prior: do not let train metadata dominate rank evidence.
prior_probs, prior_strength = _site_hour_prior_for_test(meta_test)
prior_rank = _rank_power(prior_probs, 0.96)
prior_class_mix = (0.010 + 0.026 * np.clip((support - 1.0) / 14.0, 0.0, 1.0)).astype(np.float32)
prior_class_mix[low_support] *= 0.35
prior_class_mix[perch_auc_cal_ref < 0.55] *= 0.70
prior_mix = np.clip(prior_strength.reshape(-1, 1) * prior_class_mix.reshape(1, -1), 0.0, 0.034).astype(np.float32)
final_probs = ((1.0 - prior_mix) * final_probs + prior_mix * prior_rank).astype(np.float32)

# File-level confidence scaling after consensus, then rank again for calibration stability.
n_files = len(test_paths)
pv0 = final_probs.reshape(n_files, N_WINDOWS, N_CLASSES).copy()
file_top2 = np.sort(pv0, axis=1)[:, -2:, :].mean(axis=1)
file_scale = np.clip(0.972 + 0.032 * np.sqrt(np.clip(file_top2, 0, 1)), 0.972, 1.004).astype(np.float32)
final_probs = _rank01(np.clip((pv0 * file_scale[:, None, :]).reshape(-1, N_CLASSES), 0, 1))

# Spike-preserving temporal smoothing. High-confidence isolated calls stay sharp; weaker jitter is smoothed.
pv = final_probs.reshape(n_files, N_WINDOWS, N_CLASSES).copy()
sm = pv.copy()
for i in range(1, N_WINDOWS - 1):
    nbr = 0.5 * (pv[:, i - 1] + pv[:, i + 1])
    smooth = 0.76 * pv[:, i] + 0.12 * pv[:, i - 1] + 0.12 * pv[:, i + 1]
    spike = (pv[:, i] > 0.992) | ((pv[:, i] > 0.970) & (pv[:, i] > nbr + 0.16))
    sm[:, i] = np.where(spike, pv[:, i], nbr + np.clip(smooth - nbr, -0.26, 0.26))
sm[:, 0] = np.maximum(0.94 * pv[:, 0] + 0.06 * pv[:, 1], np.where(pv[:, 0] > 0.992, pv[:, 0], 0))
sm[:, -1] = np.maximum(0.94 * pv[:, -1] + 0.06 * pv[:, -2], np.where(pv[:, -1] > 0.992, pv[:, -1], 0))
final_probs = np.clip(sm.reshape(-1, N_CLASSES), 0, 1).astype(np.float32)

probe_consensus_probs = final_probs.copy()

# V71: recompute private original anchor views in-notebook, then probability-blend with probe consensus.
def _temporal_smooth(arr, center=0.64, side=0.18, edge=0.84, clip_width=0.38):
    n_files_local = len(test_paths)
    pv_local = arr.reshape(n_files_local, N_WINDOWS, N_CLASSES).copy()
    sm_local = pv_local.copy()
    for ii in range(1, N_WINDOWS - 1):
        local = 0.5 * (pv_local[:, ii - 1] + pv_local[:, ii + 1])
        base = center * pv_local[:, ii] + side * pv_local[:, ii - 1] + side * pv_local[:, ii + 1]
        sm_local[:, ii] = local + np.clip(base - local, -clip_width, clip_width)
    sm_local[:, 0] = edge * pv_local[:, 0] + (1.0 - edge) * pv_local[:, 1]
    sm_local[:, -1] = edge * pv_local[:, -1] + (1.0 - edge) * pv_local[:, -2]
    return np.clip(sm_local.reshape(-1, N_CLASSES), 0, 1).astype(np.float32)

def _anchor_base_views():
    rel_raw_a = np.clip((perch_auc_raw_ref - 0.50) / 0.45, 0.0, 1.0)
    sed_w_v11_a = np.where(MAPPED, 0.55 - 0.20 * rel_raw_a, 0.75 - 0.25 * rel_raw_a).astype(np.float32)
    sed_w_v11_a = np.clip(sed_w_v11_a, 0.28, 0.88)
    auc_delta_a = np.clip(sed_auc_ref - perch_auc_cal_ref, -0.35, 0.35)
    support_shrink_a = np.clip((support - 1.0) / 8.0, 0.20, 1.0).astype(np.float32)
    base_sed_w_a = np.where(MAPPED, 0.48, 0.66).astype(np.float32)
    sed_w_oof_a = base_sed_w_a + 0.30 * auc_delta_a.astype(np.float32) * support_shrink_a
    sed_w_oof_a[low_support & MAPPED] = 0.44
    sed_w_oof_a[low_support & (~MAPPED)] = 0.66
    sed_w_oof_a[sed_auc_ref < 0.50] -= 0.08
    sed_w_oof_a[perch_auc_cal_ref < 0.50] += 0.08
    sed_w_oof_a = np.clip(sed_w_oof_a, 0.26, 0.82).astype(np.float32)
    sed_w_fixed_a = np.where(MAPPED, 0.46, 0.62).astype(np.float32)
    sed_w_fixed_a[low_support & MAPPED] = 0.42
    sed_w_fixed_a[low_support & (~MAPPED)] = 0.64
    blend_v11_a = _blend_rank(sed_w_v11_a)
    blend_oof_a = _blend_rank(sed_w_oof_a)
    blend_fixed_a = _blend_rank(sed_w_fixed_a)
    return blend_v11_a, blend_oof_a, blend_fixed_a, auc_delta_a, sed_w_v11_a, sed_w_oof_a, sed_w_fixed_a

blend_v11_a, blend_oof_a, blend_fixed_a, auc_delta_anchor, sw_v11_anchor, sw_oof_anchor, sw_fixed_anchor = _anchor_base_views()
anchor_v13 = (
    0.45 * _rank_power(blend_v11_a, 1.0) +
    0.35 * _rank_power(blend_oof_a, 1.0) +
    0.20 * _rank_power(blend_fixed_a, 1.0)
).astype(np.float32)
anchor_v13 = _temporal_smooth(anchor_v13, center=0.64, side=0.18, edge=0.84, clip_width=0.38)

perch_prob_anchor = _rank_power(0.65 * np.clip(perch_probs, 1e-6, 1 - 1e-6).astype(np.float32) + 0.35 * perch_rank, 1.0)
anchor_v23 = (
    0.29 * _rank_power(blend_v11_a, 1.0) +
    0.20 * _rank_power(blend_oof_a, 1.0) +
    0.13 * _rank_power(blend_fixed_a, 1.0) +
    0.38 * perch_prob_anchor
).astype(np.float32)
prior_probs_anchor, prior_strength_anchor = _site_hour_prior_for_test(meta_test)
prior_rank_anchor = _rank_power(prior_probs_anchor, 1.0)
prior_class_mix_anchor = (0.018 + 0.038 * np.clip((support - 1.0) / 12.0, 0.0, 1.0)).astype(np.float32)
prior_class_mix_anchor[low_support] *= 0.35
prior_class_mix_anchor[perch_auc_cal_ref < 0.55] *= 0.65
prior_mix_anchor = np.clip(prior_strength_anchor.reshape(-1, 1) * prior_class_mix_anchor.reshape(1, -1), 0.0, 0.052).astype(np.float32)
anchor_v23 = ((1.0 - prior_mix_anchor) * anchor_v23 + prior_mix_anchor * prior_rank_anchor).astype(np.float32)
pv_anchor = anchor_v23.reshape(len(test_paths), N_WINDOWS, N_CLASSES).copy()
file_top2_anchor = np.sort(pv_anchor, axis=1)[:, -2:, :].mean(axis=1)
file_scale_anchor = np.clip(0.965 + 0.045 * np.sqrt(np.clip(file_top2_anchor, 0, 1)), 0.965, 1.01).astype(np.float32)
anchor_v23 = _rank_power(np.clip((pv_anchor * file_scale_anchor[:, None, :]).reshape(-1, N_CLASSES), 0, 1), 1.0)
anchor_v23 = _temporal_smooth(anchor_v23, center=0.74, side=0.13, edge=0.90, clip_width=0.38)
anchor_v20 = anchor_v23.copy()
pv_fm = anchor_v23.reshape(len(test_paths), N_WINDOWS, N_CLASSES).copy()
anchor_v23 = np.clip(((1.0 - 0.04) * pv_fm + 0.04 * pv_fm.max(axis=1, keepdims=True)).reshape(-1, N_CLASSES), 0, 1).astype(np.float32)

# V86: mid-support probe rescue. v85 showed that a broad EoS/rank-power push is too
# aggressive. This keeps the conservative v84 anchor/probe base and opens a
# stronger private probe view only for classes with enough support to avoid the
# low-support false-positive regime, but not so much support that the anchor is
# already stable.
class_names = np.array([CLASS_NAME.get(label, "") for label in PRIMARY_LABELS], dtype=object)
mid_support = (support >= 5) & (support <= 150)
non_aves_mid = mid_support & (class_names != "Aves")

macro_probs = (0.4375 * anchor_v13 + 0.1500 * anchor_v23 + 0.4125 * probe_consensus_probs).astype(np.float32)
probeplus_probs = (0.3500 * anchor_v13 + 0.1200 * anchor_v23 + 0.5300 * probe_consensus_probs).astype(np.float32)
v84_like = np.where(mid_support.reshape(1, -1), probeplus_probs, macro_probs).astype(np.float32)
probe_rescue = (0.180 * anchor_v13 + 0.070 * anchor_v23 + 0.750 * probe_consensus_probs).astype(np.float32)

# V124 original EcoProto clean-blend guarded macro-risk rescue policy.
# This is a license-reduced follow-up to v103: keep the class-wise OOF/taxa/support
# rescue idea, but remove SED/cache runtime dependencies with unknown licenses.
# Rescue strength is now driven by Snowflake SED plus Perch/probe views.
base_risk = np.nan_to_num(perch_auc_cal_ref, nan=0.0).astype(np.float32)
sed_advantage = (sed_auc_ref - perch_auc_cal_ref).astype(np.float32)
texture_taxa = np.isin(class_names, ["Amphibia", "Insecta"])

protect_anchor = base_risk >= 0.85
rare_floor = support < 5
macro_weak = mid_support & (base_risk < 0.60)
macro_mild = mid_support & (base_risk >= 0.60) & (base_risk < 0.70)
texture_prior_rescue = texture_taxa & mid_support & (sed_advantage > 0.10)

rescue_mix = np.zeros(N_CLASSES, dtype=np.float32)
rescue_mix[macro_mild] = 0.12
rescue_mix[macro_weak] = 0.28
rescue_mix[macro_weak & texture_taxa] = 0.34
rescue_mix[texture_prior_rescue] = np.maximum(rescue_mix[texture_prior_rescue], 0.18)
rescue_mix[protect_anchor] = np.minimum(rescue_mix[protect_anchor], 0.05)
rescue_mix[rare_floor] = 0.0

sed_rescue_view = (0.50 * view_sedfirst + 0.30 * view_adaptive + 0.20 * probe_consensus_probs).astype(np.float32)
probe_rescue_view = (0.170 * anchor_v13 + 0.060 * anchor_v23 + 0.770 * probe_consensus_probs).astype(np.float32)
rescue_view = np.where(texture_prior_rescue.reshape(1, -1), sed_rescue_view, probe_rescue_view).astype(np.float32)

rescue_mix_2d = rescue_mix.reshape(1, -1)
positive_rescue = rescue_view > np.maximum(v84_like + 0.018, 0.48)
effective_rescue_mix = (rescue_mix_2d * np.where(positive_rescue, 1.0, 0.22)).astype(np.float32)
final_probs = ((1.0 - effective_rescue_mix) * v84_like + effective_rescue_mix * rescue_view).astype(np.float32)

def _context_mean(arr):
    pv_ctx = arr.reshape(len(test_paths), N_WINDOWS, N_CLASSES).copy()
    ctx = pv_ctx.copy()
    for ii in range(1, N_WINDOWS - 1):
        ctx[:, ii] = 0.78 * pv_ctx[:, ii] + 0.11 * pv_ctx[:, ii - 1] + 0.11 * pv_ctx[:, ii + 1]
    ctx[:, 0] = 0.90 * pv_ctx[:, 0] + 0.10 * pv_ctx[:, 1]
    ctx[:, -1] = 0.90 * pv_ctx[:, -1] + 0.10 * pv_ctx[:, -2]
    return ctx.reshape(-1, N_CLASSES).astype(np.float32)

probe_ctx = _context_mean(probe_consensus_probs)
ctx_mask = (effective_rescue_mix > 0.18) & (probe_ctx > 0.92) & (sed_rank > 0.82)
ctx_mix = np.where(ctx_mask, 0.025, 0.0).astype(np.float32)
final_probs = ((1.0 - ctx_mix) * final_probs + ctx_mix * probe_ctx).astype(np.float32)

# Row-level top-hit preservation: if v84_like has a confident local winner,
# keep that winner close to the conservative anchor even when class-level rescue
# is active. This is the main v103 guard against v102's weak top-hit proxy.
row_top = v84_like.max(axis=1, keepdims=True)
top_hit_guard = (v84_like >= (row_top - 0.018)) & (row_top >= 0.62)
final_probs = np.where(top_hit_guard, 0.96 * v84_like + 0.04 * final_probs, final_probs).astype(np.float32)

# Strong classes and rare classes keep conservative behavior; weak mid-support
# classes get a controlled rescue rather than v85-style global aggression.
protect_guard = protect_anchor.reshape(1, -1)
rare_guard = rare_floor.reshape(1, -1)
final_probs = np.where(protect_guard, 0.93 * macro_probs + 0.07 * final_probs, final_probs).astype(np.float32)
final_probs = np.where(rare_guard, 0.96 * macro_probs + 0.04 * final_probs, final_probs).astype(np.float32)

v103_counts = {
    "protect_anchor": int(protect_anchor.sum()),
    "rare_floor": int(rare_floor.sum()),
    "macro_weak": int(macro_weak.sum()),
    "macro_mild": int(macro_mild.sum()),
    "texture_prior_rescue": int(texture_prior_rescue.sum()),
    "active_rescue": int((rescue_mix > 0).sum()),
    "positive_rescue_cells": int(positive_rescue.sum()),
    "top_hit_guard_cells": int(top_hit_guard.sum()),
    "ctx_cells": int(ctx_mask.sum()),
}
print(f"V124 EcoProto clean-blend guarded macro-risk rescue counts: {v103_counts}")
print(f"V124 class rescue mix range/mean: [{rescue_mix.min():.3f}, {rescue_mix.max():.3f}] / {rescue_mix.mean():.4f}")
print(f"V124 effective rescue mix range/mean: [{effective_rescue_mix.min():.3f}, {effective_rescue_mix.max():.3f}] / {effective_rescue_mix.mean():.4f}")
print(f"V124 components: macro mean/std={macro_probs.mean():.6f}/{macro_probs.std():.6f}; v84_like mean/std={v84_like.mean():.6f}/{v84_like.std():.6f}; rescue_view mean/std={rescue_view.mean():.6f}/{rescue_view.std():.6f}; dualprobe mean/std={probe_consensus_probs.mean():.6f}/{probe_consensus_probs.std():.6f}")
print(f"OOF refs: Perch raw={perch_auc_raw_ref.mean():.3f}, Perch cal={perch_auc_cal_ref.mean():.3f}, SED={sed_auc_ref.mean():.3f}, delta={auc_delta.mean():+.3f}")
print(f"SED weights sedfirst={sed_w_sedfirst.mean():.3f}, adaptive={sed_w_adaptive.mean():.3f}, safety={sed_w_safety.mean():.3f}")
print(f"View mix: v124 = self-contained v107-like rank ceiling plus EcoProto rank-launch clean blend")
print(f"Prior strength mean={prior_strength.mean():.3f}, applied mean={prior_mix.mean():.4f}, file scale mean={file_scale.mean():.3f}")
print(f"Spike-smoothed score range: [{final_probs.min():.6f}, {final_probs.max():.6f}]")
print(f"Score mean/std: {final_probs.mean():.6f}/{final_probs.std():.6f}")

clean_guarded_base = final_probs.copy()

# V124 EcoProto rescue. This is an original, license-clean signal built only
# from competition labels plus Perch embeddings. It avoids external SED assets:
# each class gets a train-window embedding prototype; a test row is rescued only
# when prototype similarity, site/hour prior, and temporal neighborhood agree.
def _norm_rows(x):
    x = np.asarray(x, dtype=np.float32)
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-6)

def _ecoproto_rank():
    if emb_test is None:
        print("V124 EcoProto unavailable: no embedding output")
        return view_perch, np.zeros((len(meta_test), N_CLASSES), dtype=np.float32), np.zeros(N_CLASSES, dtype=bool)
    cache_pos = _cache_positions_for_full_rows()
    x_train = _norm_rows(emb_full.astype(np.float32)[cache_pos])
    x_test = _norm_rows(emb_test.astype(np.float32))
    prototypes = np.zeros((N_CLASSES, x_train.shape[1]), dtype=np.float32)
    active = np.zeros(N_CLASSES, dtype=bool)
    for ci in range(N_CLASSES):
        pos = np.flatnonzero(Y_FULL[:, ci] > 0)
        if len(pos) >= 2:
            prototypes[ci] = _norm_rows(x_train[pos].mean(axis=0, keepdims=True))[0]
            active[ci] = True
    sim = x_test @ prototypes.T
    sim[:, ~active] = np.nanmedian(sim[:, active]) if active.any() else 0.0
    sim = np.nan_to_num(sim, nan=0.0).astype(np.float32)
    centered = sim - np.median(sim, axis=0, keepdims=True)
    scaled = centered / (np.std(centered, axis=0, keepdims=True) + 1e-4)
    proto_prob = (1.0 / (1.0 + np.exp(-1.65 * np.clip(scaled, -6, 6)))).astype(np.float32)
    proto_rank = _rank_power(proto_prob, 0.72)
    print(f"V124 EcoProto active classes: {int(active.sum())}/{N_CLASSES}; sim range=[{sim.min():.4f},{sim.max():.4f}]")
    return proto_rank.astype(np.float32), proto_prob.astype(np.float32), active

proto_rank, proto_prob, proto_active = _ecoproto_rank()
eco_base = _rank_power(0.58 * proto_rank + 0.22 * prior_rank + 0.12 * view_perch + 0.08 * probe_consensus_probs, 0.82)
row_proto_peak = proto_rank.max(axis=1, keepdims=True)
row_proto_strength = np.clip((row_proto_peak - 0.58) / 0.32, 0.0, 1.0).astype(np.float32)
class_eco_gate = (
    proto_active
    & mid_support
    & (~protect_anchor)
    & (~rare_floor)
    & (perch_auc_cal_ref < 0.82)
).astype(np.float32)
class_eco_mix = (0.10 + 0.30 * np.clip((0.82 - perch_auc_cal_ref) / 0.32, 0.0, 1.0)).astype(np.float32)
class_eco_mix *= class_eco_gate
eco_mix = np.clip(row_proto_strength * class_eco_mix.reshape(1, -1), 0.0, 0.34).astype(np.float32)
final_probs = ((1.0 - eco_mix) * final_probs + eco_mix * eco_base).astype(np.float32)

# Temporal agreement rescue: raise a class only when adjacent windows also carry
# prototype evidence. This targets continuous vocal activity and avoids a global
# rank-ceiling push.
eco_pv = eco_base.reshape(len(test_paths), N_WINDOWS, N_CLASSES).copy()
eco_ctx = eco_pv.copy()
for ii in range(1, N_WINDOWS - 1):
    eco_ctx[:, ii] = 0.62 * eco_pv[:, ii] + 0.19 * eco_pv[:, ii - 1] + 0.19 * eco_pv[:, ii + 1]
eco_ctx[:, 0] = 0.86 * eco_pv[:, 0] + 0.14 * eco_pv[:, 1]
eco_ctx[:, -1] = 0.86 * eco_pv[:, -1] + 0.14 * eco_pv[:, -2]
eco_ctx = eco_ctx.reshape(-1, N_CLASSES).astype(np.float32)
temporal_mask = (eco_ctx > np.maximum(final_probs + 0.035, 0.72)) & (eco_mix > 0.06)
temporal_mix = np.where(temporal_mask, 0.13, 0.0).astype(np.float32)
final_probs = ((1.0 - temporal_mix) * final_probs + temporal_mix * eco_ctx).astype(np.float32)

# Keep v84 confident winners protected; EcoProto is a rescue sidecar, not a
# replacement for high-confidence anchor classes.
final_probs = np.where(top_hit_guard, 0.94 * v84_like + 0.06 * final_probs, final_probs).astype(np.float32)
print(f"V124 EcoProto cells: mix={int((eco_mix > 0).sum())}, temporal={int(temporal_mask.sum())}, class_gate={int(class_eco_gate.sum())}")
print(f"V124 EcoProto mix mean/max: {eco_mix.mean():.5f}/{eco_mix.max():.5f}; temporal mean={temporal_mix.mean():.5f}")
print(f"V124 post-EcoProto score range: [{final_probs.min():.6f}, {final_probs.max():.6f}]")
print(f"V124 post-EcoProto score mean/std: {final_probs.mean():.6f}/{final_probs.std():.6f}")

# V124 self-contained clean blend. First reconstruct a v107-like rank-ceiling
# branch from the pre-EcoProto guarded base, then combine it with the v109-like
# EcoProto rank-launch branch. This materializes the local probe without
# depending on prior output CSVs as Kaggle inputs.
v109_like_probs = final_probs.copy()
v107_rank_view = _rank_power(
    0.50 * view_sedfirst + 0.24 * view_perch + 0.18 * probe_consensus_probs + 0.08 * view_adaptive,
    0.62,
).astype(np.float32)
v107_restore_mask = (~rare_guard) & (~protect_guard)
v107_restore_mix = np.where(v107_restore_mask, 0.88, 0.0).astype(np.float32)
v107_like_probs = ((1.0 - v107_restore_mix) * clean_guarded_base + v107_restore_mix * v107_rank_view).astype(np.float32)
v107_top = v107_rank_view.max(axis=1, keepdims=True)
v107_top_guard = (v107_rank_view >= (v107_top - 0.030)) & (v107_top >= 0.72) & (~rare_guard)
v107_like_probs = np.where(v107_top_guard, 0.20 * v107_like_probs + 0.80 * v107_rank_view, v107_like_probs).astype(np.float32)

# The v109-like branch uses EcoProto as one term in the rank-launch view rather
# than as a standalone conservative rescue. The guard remains class-aware and
# excludes rare/protected classes.
rank_launch_view = _rank_power(
    0.42 * view_sedfirst
    + 0.18 * view_perch
    + 0.18 * probe_consensus_probs
    + 0.16 * eco_base
    + 0.06 * view_adaptive,
    0.62,
).astype(np.float32)
launch_mask = (~rare_guard) & (~protect_guard)
launch_mix = np.where(launch_mask, 0.78, 0.0).astype(np.float32)
final_probs = ((1.0 - launch_mix) * final_probs + launch_mix * rank_launch_view).astype(np.float32)

rank_top = rank_launch_view.max(axis=1, keepdims=True)
rank_top_guard = (rank_launch_view >= (rank_top - 0.030)) & (rank_top >= 0.72) & (~rare_guard)
final_probs = np.where(rank_top_guard, 0.22 * final_probs + 0.78 * rank_launch_view, final_probs).astype(np.float32)
v109_like_probs = final_probs.copy()

final_probs = (0.40 * v107_like_probs + 0.60 * v109_like_probs).astype(np.float32)
print(f"V124 v107-like cells: restore={int(v107_restore_mask.sum())}, top_guard={int(v107_top_guard.sum())}")
print(f"V124 EcoProto rank-launch cells: launch={int(launch_mask.sum())}, top_guard={int(rank_top_guard.sum())}")
print(f"V124 clean blend weights: v107_like=0.40, v109_like=0.60")
print(f"V124 post-cleanblend score range: [{final_probs.min():.6f}, {final_probs.max():.6f}]")
print(f"V124 post-cleanblend score mean/std: {final_probs.mean():.6f}/{final_probs.std():.6f}")


# V124 post-final lightweight sidecar.  This is intentionally cheaper than
# recomputing a Tsubasa final layer: it compares ranks against the clean final
# output and only nudges selected classes where Tsubasa is ahead.
v124_clean_final_probs = final_probs.copy()
v124_clean_final_rank = _rank01(v124_clean_final_probs)
v124_postfinal_gate = (
    v124_selected_mask.reshape(1, -1)
    & (v124_tsubasa_rank > (v124_clean_final_rank + 0.02))
)
v124_postfinal_mix = np.where(v124_postfinal_gate, 0.30, 0.0).astype(np.float32)
final_probs = ((1.0 - v124_postfinal_mix) * v124_clean_final_probs + v124_postfinal_mix * v124_tsubasa_rankcal).astype(np.float32)
v124_postfinal_summary = pd.DataFrame(
    [
        {
            "branch": "clean_final",
            "role": "base",
            "min": float(v124_clean_final_probs.min()),
            "max": float(v124_clean_final_probs.max()),
            "mean": float(v124_clean_final_probs.mean()),
            "std": float(v124_clean_final_probs.std()),
        },
        {
            "branch": "tsubasa_rankcal_sidecar",
            "role": "postfinal_sidecar",
            "min": float(v124_tsubasa_rankcal.min()),
            "max": float(v124_tsubasa_rankcal.max()),
            "mean": float(v124_tsubasa_rankcal.mean()),
            "std": float(v124_tsubasa_rankcal.std()),
        },
        {
            "branch": "v124_postfinal_blend",
            "role": "submission",
            "min": float(final_probs.min()),
            "max": float(final_probs.max()),
            "mean": float(final_probs.mean()),
            "std": float(final_probs.std()),
        },
    ]
)
v124_postfinal_summary.to_csv("v124_postfinal_tsubasa_summary.csv", index=False)
print(f"V124 selected sidecar classes: {v124_selected_classes}")
print(f"V124 post-final rank-margin gate cells: {int(v124_postfinal_gate.sum())} / {v124_postfinal_gate.size}")
print("V124 final layer mode: clean final once plus post-final rank-calibrated sidecar")
print(f"V124 final score range: [{final_probs.min():.6f}, {final_probs.max():.6f}]")
print(f"V124 final score mean/std: {final_probs.mean():.6f}/{final_probs.std():.6f}")








In [ ]:
submission = pd.DataFrame(final_probs, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", meta_test["row_id"].values)
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)
assert not submission.isna().any().any()
assert len(submission) == len(test_paths) * N_WINDOWS
submission.to_csv("submission.csv", index=False)
print(f"Saved: {submission.shape}")
print(submission.head(3))